# **Data Versioning**

---

## Why Version Data?

Code is not enough to reproduce an ML experiment. The **same code** on a **different dataset** produces a different model. Yet datasets are too large to commit to Git, mutate over time (new labels, cleaned rows, added samples), and are often pulled from object storage or labeling tools.

Data versioning answers a single question: *given a commit, exactly which bytes of data produced this model?* Without it you cannot:

* reproduce a result or a bug,
* roll back to the dataset that trained a known-good model,
* audit what data a deployed model ever saw.

The core idea is the same as Git for code: track an **immutable, content-addressed snapshot** of the data and pin the experiment to it.

## The Tools

| Tool | Model | Best for |
|------|-------|----------|
| **DVC** | Git stores small `.dvc` pointer files (an md5 hash + size); the actual data lives in a remote cache (S3, GCS, Azure, SSH, local). | Versioning datasets/models alongside Git; pipelines (`dvc.yaml`). |
| **Git-LFS** | Replaces large files in Git with text pointers; binaries pushed to an LFS server. | A handful of medium binaries (weights, images) inside a normal Git workflow. |
| **lakeFS** | Git-like branch/commit/merge semantics **on top of an object store** (S3). | Petabyte-scale data lakes, atomic branching of whole buckets. |

Complementary concepts:

* **Dataset hashing / manifests** — record a content hash (e.g. md5/sha256) per file, plus row counts and class distribution, in a `manifest.json` checked into Git. This makes a dataset *self-describing* and lets CI assert the data has not silently changed.
* **Reproducibility** — pin the data version (a DVC/Git commit hash) in the same commit as the training config so the pair `(code, data)` is fully determined.

## DVC: Minimal Workflow

DVC stores a tiny `.dvc` pointer in Git and the heavy bytes in a remote.

```bash
# one-time setup inside an existing git repo
pip install "dvc[s3]"
dvc init
dvc remote add -d storage s3://my-bucket/dvc-cache

# start tracking a dataset
dvc add data/train.csv          # creates data/train.csv.dvc (md5 + size)
git add data/train.csv.dvc data/.gitignore
git commit -m "Track training data v1"

# push the actual bytes to the remote, then anyone can reproduce:
dvc push

# on another machine / later checkout
git checkout <commit>
dvc pull                        # restores the exact bytes for that commit
```

The `.dvc` file is plain YAML, e.g.:

```yaml
outs:
- md5: a1b2c3d4e5f6...
  size: 10485760
  path: train.csv
```

Because the hash lives in Git, `git checkout <old-commit> && dvc pull` always returns the dataset that commit was built on.

## A Lightweight Manifest (no extra tooling)

If you only need to *detect* data drift in CI, a hashed manifest is often enough:

```python
import hashlib, json, pathlib

def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

root = pathlib.Path("data")
manifest = {
    str(p.relative_to(root)): file_md5(p)
    for p in sorted(root.rglob("*")) if p.is_file()
}
json.dump(manifest, open("manifest.json", "w"), indent=2)
```

Commit `manifest.json`; a CI step re-computes it and fails if the hashes differ from what the code expects.

## References

* DVC documentation: https://dvc.org/doc
* DVC `add` / `push` / `pull`: https://dvc.org/doc/start/data-management/data-versioning
* Git-LFS: https://git-lfs.com
* lakeFS: https://docs.lakefs.io